# Training a MACE model


In this notebook, the flow of training a MACE model is detailed step by step.



In [15]:
import matplotlib.pyplot    as plt
plt.ioff()
import numpy                as np
import torch
from time                   import time
import datetime             as dt
from tqdm                   import tqdm

from pathlib import Path
import sys

parentpath = str(Path().cwd())+'/'

print(parentpath)

sys.path.append(parentpath)

import src.mace.CSE_0D.dataset  as ds
import src.mace.CSE_0D.plotting as plotting
import src.mace.train           as train
import src.mace.test            as test
import src.mace.load            as load
import src.mace.mace            as mace
from src.mace.loss              import Loss
import src.mace.loss            as loss
import src.mace.utils           as utils
from src.mace.meta              import Meta

specs_dict = utils.get_specs()

%reload_ext autoreload
%autoreload 2

c:\Users\bartd\Documents\Astronomy\Thesis\MACE/


## Setting up

>***Note***:
>If you are training the architecture with your own data, you can set up the model in the following way. However, if you will follow the notebook, we will load a pre-trained model to continue the training, just for tutorial purposes. Then, skip the cells on "Set up the model".


### Naming the model
The default model to name the model is with the date and time of training. This can be adjusted with changing the parameter name.




In [16]:
now = dt.datetime.now()
name = str(now.strftime("%Y%m%d")+'_'+now.strftime("%H%M%S"))
name = "example_out_v2/" + name

path = parentpath + 'model/'+name

print(path)

c:\Users\bartd\Documents\Astronomy\Thesis\MACE/model/example_out_v2/20261111_172112


Make directories to store output and trained model.



In [17]:
utils.makeOutputDir(path)
utils.makeOutputDir(path+'/nn')
utils.makeOutputDir(path+'/figs')

'c:\\Users\\bartd\\Documents\\Astronomy\\Thesis\\MACE/model/example_out_v2/20261111_172112/figs'

### Reading input file
Fill in the name of the input file for arg.

For tutorial purposes, the amount of training data and training epochs is limited in this example.

In [18]:
arg = 'example_v2'

In [19]:
infile = parentpath+'input/'+arg+'.in'
meta = Meta(infile)
metadata = meta.make_meta(path)
meta.print()

------------------------------
Name: Unnamed
------------------------------
     inputfile: c:\Users\bartd\Documents\Astronomy\Thesis\MACE/input/example_v2.in
    #n, #p, #z: 468 4 8
   Autoencoder: simple
      Neural G: False
 Learning Rate: 0.0001
       #Epochs: 150
   #Evolutions: 16
Train|Validate: 10|3
      Datatype: 1DCSE
     Loss Type: abs_idn
  Loss Weights: {'abs': 1.0, 'idn': 1.0}



### Set up Pytorch

In [20]:
cuda   = False
DEVICE = torch.device("cuda" if cuda else "cpu")
batch_size = 1
kwargs = {'num_workers': 1, 'pin_memory': True}

### Set up the model


Make the model, randomly initialised.



In [21]:
model = mace.MACE(meta.n_dim, meta.p_dim, meta.z_dim,
                  ae_type=meta.ae_type, g_nn=meta.g_nn, 
                  atol=meta.atol, rtol=meta.rtol)

num_params = utils.count_parameters(model)
print(f'\nThe model has {num_params} trainable parameters')


The model has 284692 trainable parameters


### Load training & validation data

Load the train, validation, and test data sets.

In [39]:
for n, p, dt in data_loader:
    print("datapoint:")
    print(n.shape)
    print(p.shape)
    print(dt.shape)

c:\Users\bartd\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


datapoint:
torch.Size([1, 135, 468])
torch.Size([1, 134, 4])
torch.Size([1, 134])
datapoint:
torch.Size([1, 135, 468])
torch.Size([1, 134, 4])
torch.Size([1, 134])
datapoint:
torch.Size([1, 81, 468])
torch.Size([1, 80, 4])
torch.Size([1, 80])
datapoint:
torch.Size([1, 135, 468])
torch.Size([1, 134, 4])
torch.Size([1, 134])
datapoint:
torch.Size([1, 135, 468])
torch.Size([1, 134, 4])
torch.Size([1, 134])
datapoint:
torch.Size([1, 135, 468])
torch.Size([1, 134, 4])
torch.Size([1, 134])
datapoint:
torch.Size([1, 135, 468])
torch.Size([1, 134, 4])
torch.Size([1, 134])


In [22]:
traindata, testdata, data_loader, test_loader = ds.get_data(dt_fract=meta.dt_fract,
                                                            nb_samples=meta.nb_train, batch_size=batch_size,
                                                            nb_test=meta.nb_valid,kwargs=kwargs, inpackage=True, data_type=meta.data_type)

> Train state: True
Found paths: 13
[12  5  0  3 11  3  7  9  3  5]
number of test paths: 3
Selected test paths: 3
Selected paths: 7


> Train state: False
Found paths: 13
[12  5  0  3 11  3  7  9  3  5]
number of test paths: 3
Selected test paths: 3
Selected paths: 3


Dataset:
------------------------------
  total # of samples: 10
#   training samples: 7
# validation samples: 3
               ratio: 0.3
     #  test samples: 3


c:\Users\bartd\Documents\Astronomy\Thesis\MACE\src\mace\CSE_0D\dataset.py:262: UserWarning: Input line 14 contained no data and will not be counted towards `max_rows=50000`.  This differs from the behaviour in NumPy <=1.22 which counted lines rather than rows.  If desired, the previous behaviour can be achieved by using `itertools.islice`.
Please see the 1.23 release notes for an example on how to do this.  If you wish to ignore this warning, use `warnings.filterwarnings`.  This warning is expected to be removed in the future and is given only once per `loadtxt` call.
  paths = np.loadtxt(loc + 'data/paths_train_data.txt', dtype=str)


## Loading the pre-trained model


Setting the directory of the model.



In [23]:
outloc  = parentpath+'model/'
dirname = 'example_weights_v2'
epoch   = 7

You can load a MACE model from a directory to get a solver, loss objects and the meta file.

In [24]:
model, losses, metadata = load.load_MACE(f'model/{dirname}/', epoch=epoch)
print(f'The model has {utils.count_parameters(model)} trainable parameters')

The model has 284692 trainable parameters


Alternatively, you can overwrite the weights of an initialised model.

In [25]:
utils.overwrite_weights(model, f'model/{dirname}/', epoch=epoch)

## Training the model

Training the model in [Maes et al. (2024)](https://ui.adsabs.harvard.edu/abs/2024arXiv240503274M/abstract) happens it two stages:

1. For the first few epochs (meta.ini_epochs) the models is trained with unscaled losses.
2. The losses of the model are rescaled, normalised based in the losses in the first few epochs.

### Stage 1
Initialise loss.

In [ ]:
trainloss = loss.setup_loss(meta.lossweights)
validloss = loss.setup_loss(meta.lossweights)

Train the model & time the training.

In [ ]:
lr = meta.lr
lr = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
train.train(model,
            data_loader, test_loader,
            trainloss, validloss,
            optimiser=optimizer, 
            nb_epochs=meta.ini_epochs,
            nb_evol=meta.nb_evol,
            batch_size=8)

In [ ]:
loss.plot({'train':trainloss, 'valid':validloss})

### Stage 2
Normalise losses.

In [ ]:
trainloss.normalize()
validloss.normalize()

In [ ]:
lr = meta.lr
lr = 1e-3 # violence
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

Continue the training. Use lower batch count initially.


In [ ]:
train.train(model,
            data_loader, test_loader,
            trainloss, validloss,
            optimiser=optimizer, 
            nb_epochs=30,
            nb_evol=meta.nb_evol,
            batch_size=8)

### Stage 3

Normalize losses again.

In [ ]:
trainloss.normalize()
validloss.normalize()

In [ ]:
lr = meta.lr
lr = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

Continue training with full batches.

In [ ]:
train.train(model,
            data_loader, test_loader,
            trainloss, validloss,
            optimiser=optimizer, 
            nb_epochs=50,
            nb_evol=meta.nb_evol)

In [ ]:
loss.plot({'train':trainloss, 'valid':validloss})

## Saving the model
Save the losses & specifics of the data set.

In [ ]:
print('path:', path)

trainloss.save(path+'/train')
validloss.save(path+'/valid')

#min_max = np.stack((traindata.mins, traindata.maxs), axis=1)
#np.save(path+'/minmax', min_max)

Save the model and the status of the solver (see [torchode status](https://github.com/martenlienen/torchode/blob/main/torchode/status_codes.py) for more details).

In [ ]:
torch.save(model.state_dict(),path+'/nn/nn.pt')

#np.save(path+'/train/status', model.get_status('train')) # type: ignore
#np.save(path +'/valid/status', model.get_status('test') ) # type: ignore

Plot the evolution of the loss and save the figure.



In [ ]:
plt.close()
loss.plot({'train':trainloss, 'valid':validloss})
plt.savefig(path+'/figs/loss.png')
plt.show()

## Testing the model


Perform tests on the trained model to get an accuracy indication, according to the following error metric:

$$ {\rm error} = \frac{\log_{10}\boldsymbol{n}-\log_{10}\hat{\boldsymbol{n}}}{\log_{10}\boldsymbol{n}}, $$

which is executed element-wise and subsequently summed over the different chemical species

### Performing the tests
(tbd)

In [ ]:
sum_err_step = 0
sum_err_evol = 0

step_calctime = list()
evol_calctime = list()

for i in tqdm(range(len(traindata.testpath))):
#     print(i+1,end='\r')
    testpath = traindata.testpath[i]

    err_test, err_evol, step_time, evol_time, n, n_hat, n_evol = test.test_model('1DCSE', model,testpath, meta, printing = False, inpackage=True, datapath='train' )

    sum_err_step += err_test
    sum_err_evol += err_evol

    step_calctime.append(step_time)
    evol_calctime.append(evol_time)

### Saving the outcome of the tests
The error saved is normalised over the amount of tests, given by len(traindata.testpath).

(tbd)

### Plotting abundances

In [ ]:
testpath = 'mid_dens'
model1D, (n, p, dt), info = ds.get_test_data('1DCSE', testpath, metadata, inpackage=True)

In [ ]:
tic = time()
with torch.no_grad():
    n_hat, status = model.rollout(n[0:1], p[None], dt[None])
toc = time()

n_evol = torch.cat((n[0:1], n_hat[0]), dim=0)      ## (N, 468)

print(f'Evolution of {n_hat.shape[1]} steps in {toc-tic:.2f} s')
print('Solver status counts [success, general error, dt_min, max_steps, inf. norm]:',
      torch.bincount(status.flatten(), minlength=5).tolist())

In [ ]:
n_real = ds.get_abs(n.numpy())
n_evol = ds.get_abs(n_evol.numpy())

err, err_evol = utils.error(n_real, n_evol)

print('Evolution error (Eq. 23):', np.round(err_evol, 3))

In [ ]:
fig12_specs = ['CO', 'H2O', 'OH', 'C2H2', 'C2H', 'CH3C5NH+', 'C10H2+']

fig, axes = plotting.plot_abs(model1D, '1DCSE', n_real, n_evol, specs=fig12_specs, step=False)

#axes[2].set_ylim(-.5,.1)
fig.suptitle(f'MACE v2, {testpath}', y=1.0)
fig.savefig(path+'/figs/preds.png')
plt.show()